# 2주차: Tensor와 DataLoader

## 오늘의 목표

이번 시간에는 손글씨 숫자 데이터를 PyTorch가 사용할 수 있는 형태로 바꿔 봅니다.

- NumPy 배열과 PyTorch Tensor의 차이를 알아봅니다.
- 이미지의 `shape`과 `dtype`을 확인합니다.
- 이미지와 label을 Dataset으로 묶습니다.
- DataLoader로 데이터를 batch 단위로 꺼냅니다.

> 핵심 흐름: 이미지와 label → Tensor → Dataset → DataLoader → batch

## 1. 지난 시간 복습

지난 시간에 사용한 `load_digits` 데이터셋을 다시 불러옵니다.

다음 질문에 먼저 답해 보세요.

1. `images`에는 무엇이 들어 있나요?
2. `labels`에는 무엇이 들어 있나요?
3. `images[0]`과 `labels[0]`은 어떤 관계인가요?

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()

images = digits.images
labels = digits.target

print('images shape:', images.shape)
print('labels shape:', labels.shape)
print('첫 번째 label:', labels[0])

## 2. NumPy 배열에서 Tensor로

PyTorch에서는 모델 학습에 사용할 숫자 데이터를 `Tensor` 형태로 다룹니다.
NumPy 배열과 비슷하지만 PyTorch 계산과 모델에 바로 사용할 수 있는 자료형입니다.

먼저 이미지 한 장을 Tensor로 바꿔 보세요.

In [ ]:
import torch

image_numpy = images[0]
image_tensor = torch.tensor(image_numpy)

print('NumPy type:', type(image_numpy))
print('Tensor type:', type(image_tensor))
print('Tensor shape:', image_tensor.shape)
print('Tensor dtype:', image_tensor.dtype)

### 생각해 보기

- Tensor의 shape은 무엇을 의미하나요?
- 이미지 한 장은 몇 개의 픽셀을 가지고 있나요?
- AI 모델의 입력으로 사용하려면 숫자 데이터가 필요할까요, 이미지 파일이 필요할까요?

## 3. Tensor의 dtype 확인하기

이미지는 계산에 사용할 숫자이므로 보통 실수형 Tensor로 변환합니다.
label은 클래스 번호이므로 정수형으로 보관합니다.

다음 코드에서 `???`를 알맞게 바꿔 보세요.

In [ ]:
image_tensor = torch.tensor(
    image_numpy,
    dtype=???
)

print(image_tensor.dtype)

## 4. 전체 데이터를 Tensor로 변환하기

다음 코드를 실행하기 전에 결과를 예상해 보세요.

- `X`의 shape은 무엇일까요?
- `y`의 shape은 무엇일까요?
- `X`와 `y`의 dtype은 각각 무엇이어야 할까요?

In [ ]:
# 아래 코드의 ???를 완성해 보세요.
X = torch.tensor(digits.images, dtype=???)
y = torch.tensor(digits.target, dtype=???)

print('X shape:', X.shape)
print('y shape:', y.shape)
print('X dtype:', X.dtype)
print('y dtype:', y.dtype)

## 5. 픽셀 값 정규화

현재 픽셀 값은 0부터 16까지입니다. 이를 0부터 1 사이의 값으로 바꿔 보겠습니다.

정규화는 데이터의 숫자 범위를 일정하게 맞추는 과정입니다.

In [ ]:
print('변환 전 최솟값:', X.min().item())
print('변환 전 최댓값:', X.max().item())

X_normalized = X / ???

print('변환 후 최솟값:', X_normalized.min().item())
print('변환 후 최댓값:', X_normalized.max().item())

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(6, 3))

axes[0].imshow(X[0], cmap='gray')
axes[0].set_title('원본')
axes[0].axis('off')

axes[1].imshow(X_normalized[0], cmap='gray')
axes[1].set_title('0~1 정규화')
axes[1].axis('off')

plt.tight_layout()
plt.show()

### 관찰 기록

두 이미지가 눈으로 보기에 어떻게 다른가요?

- 화면에서 보이는 모양:
- 내부 숫자 값의 변화:
- 정규화가 필요한 이유에 대한 내 생각:

## 6. Dataset 만들기

`TensorDataset`은 이미지와 label을 같은 위치의 데이터끼리 묶어 줍니다.

예를 들어 `dataset[0]`은 다음과 같은 한 쌍입니다.

```text
(첫 번째 이미지, 첫 번째 정답 label)
```

In [ ]:
from torch.utils.data import TensorDataset

dataset = TensorDataset(X_normalized, y)

print('데이터 개수:', len(dataset))
sample_image, sample_label = dataset[0]
print('sample image shape:', sample_image.shape)
print('sample label:', sample_label)

### 생각해 보기

- `len(dataset)`은 무엇을 의미하나요?
- `dataset[0]`에는 무엇이 들어 있나요?
- `dataset[0]`과 `dataset[1]`의 label은 같을까요?

## 7. DataLoader 만들기

DataLoader는 Dataset에서 데이터를 여러 개씩 꺼내 batch로 만들어 줍니다.

아래 코드에서 `batch_size`를 바꿔 보고 shape 변화를 관찰해 보세요.

In [ ]:
from torch.utils.data import DataLoader

loader = DataLoader(
    dataset,
    batch_size=???,
    shuffle=True
)

images_batch, labels_batch = next(iter(loader))

print('images batch shape:', images_batch.shape)
print('labels batch shape:', labels_batch.shape)

### batch shape 예측하기

`batch_size=8`이라면 이미지 batch의 shape은 어떻게 될까요?

```text
이미지 한 장: (8, 8)
이미지 여러 장: (____, 8, 8)
label 여러 개: (____,)
```

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(8, 4))

for image, label, ax in zip(
    images_batch,
    labels_batch,
    axes.flatten()
):
    ax.imshow(image, cmap='gray')
    ax.set_title(f'label: {label.item()}')
    ax.axis('off')

plt.tight_layout()
plt.show()

## 8. 혼자 해보기

### 도전 1: batch size 바꾸기
`batch_size`를 4, 16, 32로 바꾸고 다음 표를 완성해 보세요.

| batch size | images shape | labels shape |
|---:|---|---|
| 4 |  |  |
| 16 |  |  |
| 32 |  |  |

### 도전 2: shuffle 비교
`shuffle=True`와 `shuffle=False`일 때 첫 번째 batch의 label 순서를 비교해 보세요.

### 도전 3: 특정 숫자만 보기
`target_number`를 0, 3, 7로 바꾸면서 해당 숫자의 이미지를 찾아 출력해 보세요.

### 도전 4: 나만의 질문 만들기
오늘 배운 데이터 구조를 활용해 스스로 실험 질문을 하나 만들고 답해 보세요.

## 9. 오늘의 정리

다음 문장을 자신의 말로 완성해 보세요.

- Tensor는 ____________________________________________이다.
- Dataset은 ____________________________________________이다.
- DataLoader는 _________________________________________이다.
- batch size가 커지면 batch의 첫 번째 shape은 ________________.
- 오늘 가장 새롭게 알게 된 점은 _______________________________.